# Testes e validação

Conferências do grafo e das estratégias (distâncias conhecidas, simetria, desigualdade triangular e Held-Karp contra força bruta) e validação da geometria contra a calibração feita na modelagem dos layouts.

In [1]:
%run grafo.ipynb
%run rotas.ipynb

import itertools
import random

## Testes do grafo e das estratégias

1. A matriz $D$ tem o tamanho certo, é simétrica, tem diagonal zero e reproduz distâncias calculadas à mão (ENTRADA–CAIXA = 64 m; mesma posição em corredores vizinhos = 1,25 + 4 + 1,25 = 6,5 m).
2. A desigualdade triangular vale para todos os trios de vértices; se falhasse, o Dijkstra teria deixado algum caminho maior que um desvio.
3. Os três layouts ocupam exatamente os 134 slots, e os 100 pedidos têm seções.
4. O Held-Karp empata com a força bruta (todas as permutações) em pedidos de 1 a 6 seções.
5. Em 30 pedidos sorteados, toda estratégia visita cada slot uma vez e `custoMin` nunca anda mais que as outras.

In [2]:
assert distancias.shape == (136, 136)
assert np.allclose(distancias, distancias.T) and np.all(np.diag(distancias) == 0)
assert distancias[loja.ENTRADA, loja.CAIXA] == 64.0
assert distancias[loja.ENTRADA, loja.indice_slot(1, 1)] == 1.25
assert distancias[loja.indice_slot(1, 1), loja.indice_slot(2, 1)] == 6.5

assert np.all(distancias[:, None, :] <= distancias[:, :, None] + distancias[None, :, :] + 1e-9)

for nome in LAYOUTS:
    assert set(carregar_layout(nome).values()) == set(loja.slots)
assert len(pedidos) == 100 and pedidos["secoes"].notna().all()

sorteio = random.Random(42)
for n in range(1, 7):
    slots = sorteio.sample(range(loja.n_slots), n)
    forca_bruta = min(custo_rota(ordem, distancias, loja) for ordem in itertools.permutations(slots))
    assert abs(custo_rota(held_karp(slots, distancias, loja), distancias, loja) - forca_bruta) < 1e-9

for _ in range(30):
    slots = sorteio.sample(range(loja.n_slots), sorteio.randint(1, 18))
    rotas = {estrategia: executar(estrategia, slots, distancias, loja) for estrategia in ESTRATEGIAS}
    assert all(sorted(ordem) == sorted(slots) for ordem, _ in rotas.values())
    assert rotas["custoMin"][1] <= min(metros for _, metros in rotas.values()) + 1e-9

print(f"Grafo com {len(loja.vizinhos)} vértices, {loja.n_slots} slots e {len(pedidos)} pedidos: verificações ok")

Grafo com 304 vértices, 134 slots e 100 pedidos: verificações ok


## Validação da geometria

Refaz a distância média dos 1.000 pedidos de validação em L0 e L1 (n_min = 300) e compara com a calibração das modeladoras, que usou Held-Karp até 15 seções e vizinho mais próximo + 2-opt acima disso.

In [3]:
amostra = pd.read_csv(PASTA_DADOS / "amostras" / "amostra_validacao_1000.csv")
pedidos_validacao = [[int(secao) for secao in str(texto).split(";")] for texto in amostra["aisles"]]
calibracao = pd.read_csv(PASTA_DADOS / "amostras" / "calibracao_n_min.csv").set_index("n_min").loc[300]
esperado = {"L0": calibracao["dist_L0_validacao_m"], "L1": calibracao["dist_media_validacao_m"]}

linhas = []
for layout, alvo in esperado.items():
    posicoes = carregar_layout(layout)
    metros = []
    heuristicos = 0
    for secoes in pedidos_validacao:
        slots = slots_do_pedido(secoes, posicoes, loja)
        if usa_held_karp(slots):
            ordem = held_karp(slots, distancias, loja)
        else:
            heuristicos += 1
            ordem = dois_opt(vizinho_mais_proximo(slots, distancias, loja), distancias, loja)
        metros.append(custo_rota(ordem, distancias, loja))

    media = float(np.mean(metros))
    linhas.append({"layout": layout, "distancia_obtida_m": media, "distancia_calibracao_m": alvo,
                   "diferenca_m": media - alvo, "pedidos_heuristicos": heuristicos,
                   "dentro_da_tolerancia": abs(media - alvo) <= 0.05})

tabela = pd.DataFrame(linhas)
PASTA_RESULTADOS.mkdir(exist_ok=True)
tabela.to_csv(PASTA_RESULTADOS / "validacao_geometria.csv", index=False, float_format="%.3f")
tabela

,layout,distancia_obtida_m,distancia_calibracao_m,diferenca_m,pedidos_heuristicos,dentro_da_tolerancia
0,L0,157.2445,157.234,0.0105,76,True
1,L1,144.7640,144.783,-0.0190,76,True
